<a id="unit-u.d48c9975b3a44f898d7c0e93dbf4bd8b"></a>

<div>
  <div>10-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Ұсыну кестесін жасау және салыстырып тексеру, төлемдерді өлшеу</div>
</div>

Метрика SQL-ден бұрын келісімшарт болады. Күндік тапсырыстар мен сомасының келісімшартын жазып, 9-зертханалық жұмыстың тапсырыс фактісінен тапсырыс күні мен аймағы түйіршіктілігінде `daily_order_metrics_l2` ұсыну кестесін жасайсыз, оны егжей-тегжейлі деректермен толық сыртқы біріктіру арқылы салыстырасыз: ол ұсыну кестесі жоғалтып, ішкі біріктіру байқамаған күнді табады. Нақты күндер мен аймақтарда жинақтаушы терезенің түйіршіктілігін сақтайтынын, орташалардың орташасы қателесетінін көресіз. Содан кейін мысал төлемдерін өлшейсіз: терезедегі төленген GMV, сол терезеде жасалған тапсырыстар конверсиясы, қайта сатып алушылар; нақты және жуық бөлек мәндер санын салыстырасыз; шағын кестенің аз жол оқып, егжей-тегжейлі деректердей жауап беруін тексересіз. Шамамен 90 минут жоспарлаңыз.

**10-модуль: метрикаларды өңдеу және ұсыну** · [Сабақ](course.md) · [Тест](quiz10_metric_processing.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.ba09b3460f6b43c0b5c98abec62bf9ff"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Алдымен сол дерекқорда 9-зертханалық жұмысты аяқтаңыз. Бұл жұмыс оның `orders_fact_l2` тапсырыс фактісін және мысалдың `dwd_orders_l2`, `dwd_payments_l2`, `customers_case_dim_l2` кестелерін оқиды, ешқайсысын өзгертпейді; біреуі жоқ болса, баптау ұяшығы тоқтап, хабарлайды.

Жұмыс 11-жұмыс жариялайтын `daily_order_metrics_l2`; мысал үшін 11-жұмыстың қосымша бақылау тақтасы оқитын `dwd_refunds_l2`, `paid_order_metrics_l2` және `bi_paid_kpis_l2` көрінісі; 8-бөлім өлшеуі үшін `perf_orders_detail_l2`, `perf_daily_orders_l2` қайта жасайды. Бәрі зертханалық дерекқорыңызда қалады. Бәрі курс сынақ ортасында орындалады.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, normalized
from dw_course.ui import show_records, show_records, show_sql
from dw_course.warehouse import case_fixture, compare_query_profiles

lab = connect_sandbox(module=10)

<a id="unit-u.0538d2da48d54fc58b7462ceef92270a"></a>

<a id="1-write-the-metric-contract-then-build-the-service-table"></a>

## 1. Метрика келісімшартын жазу, содан кейін ұсыну кестесін жасау

«Күндік тапсырысқа» SQL-ден бұрын жауаптар қажет. Осы жұмыстың келісімшарты:

| Тармақ | Күндік тапсырыстар және тапсырыс сомасы |
| --- | --- |
| Жиын | `orders_fact_l2` ішіндегі CANCELLED емес тапсырыстар |
| Түйіршіктілік | Әр тапсырыс күні мен аймағына бір жол |
| Көрсеткіштер | `order_count` — тапсырыс саны; `gross_amount` — алынған ақша емес, тапсырыстың салыққа дейінгі құны `order_amount` қосындысы |
| Уақыт | `order_date` — тапсырыстың 1-нұсқа оқиғасының күні, сеанстың +08:00 уақыт белдеуінде |
| Қатынас | Орташа тапсырыс сомасы = `gross_amount` / `order_count`, сұралған түйіршіктілік қосындыларынан есептеледі; тапсырыс болмаса NULL |

`daily_order_metrics_l2` ұсыну кестесі көрсеткіштерді сол түйіршіктілікте сақтайды. Ол — Aggregate Key кестесі: бірдей кілтті жолдар қосылады, сондықтан жолдар келгенше `order_count` пен `gross_amount` өсе береді. Бұл жаңа тапсырыс үшін дұрыс, бір пакетті екі рет жүктеуге қате, сондықтан ұяшық әр орындауда кестені егжей-тегжейлі деректерден қайта құрады.


In [ ]:
lab.execute("DROP TABLE IF EXISTS daily_order_metrics_l2")
service_ddl = """CREATE TABLE daily_order_metrics_l2 (
    order_date DATE NOT NULL,
    region VARCHAR(16) NOT NULL,
    order_count BIGINT SUM NOT NULL DEFAULT "0",
    gross_amount DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(order_date, region)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
service_load = """INSERT INTO daily_order_metrics_l2
SELECT order_date, region, COUNT(*), SUM(order_amount)
FROM orders_fact_l2
WHERE status <> 'CANCELLED'
GROUP BY order_date, region"""
show_sql("The service table and its load", service_ddl + ";\n\n" + service_load + ";")
lab.execute(service_ddl)
lab.execute(service_load)
lab.sql("SELECT * FROM daily_order_metrics_l2 ORDER BY order_date, region", title="The service table")
expect(lab.query("SELECT CAST(order_date AS STRING), region, order_count, gross_amount FROM daily_order_metrics_l2 ORDER BY order_date, region"),
       [("2026-01-01", "EAST", 5, "720.00"), ("2026-01-01", "WEST", 4, "480.00"), ("2026-01-02", "EAST", 1, "110.00")],
       title="Three rows at the grain of the contract")

<a id="unit-u.f002a7d3825b4a3397bf9fa115281911"></a>

**Күтілетін нәтиже**

Анықтама және жүктеу, содан кейін үш жол: 2026-01-01 EAST — 5 тапсырыс, 720.00; 2026-01-01 WEST — 4, 480.00; 2026-01-02 EAST — 1, 110.00, жасыл белгі. Бірге 10 тапсырыс және 1310.00, 8-жұмыс көрінісінің тапсырыстары.

Кесте орташаларды емес, сандар мен қосындыларды сақтайды: әр жолда сақталған орташаны күн не бүкіл кезеңге қосуға болмайды. Сол пакетті екі рет жүктеу 1 қаңтар EAST үшін 10 тапсырыс және 1440.00 көрсетер еді; оның орнына ұяшық кестені жойып, қайта құрады.


<a id="unit-u.dff9db9679b749f096f2be625bd3b0bc"></a>

<a id="2-roll-the-metric-up-for-a-dashboard"></a>

## 2. Метриканы бақылау тақтасы үшін ірілендіру

Бақылау тақтасы әр күнге бір жол көрсетеді. Әр күннің ұсыну жолдарын қосып, орташаны қосындылардан есептейді, сондықтан бір күнде қанша аймақ болса да, орташа дұрыс қалады. Шартты қосынды бір аймақты жалпы мәнмен сол жолда қатар көрсетеді: `SUM(CASE WHEN region = 'EAST' THEN order_count ELSE 0 END)` әр топтың тек EAST жолдарын қосады.


In [ ]:
dashboard_sql = """SELECT CAST(order_date AS STRING) AS order_date,
       SUM(CASE WHEN region = 'EAST' THEN order_count ELSE 0 END) AS east_orders,
       SUM(order_count) AS all_orders,
       SUM(gross_amount) AS gross_amount,
       CAST(SUM(gross_amount) / SUM(order_count) AS DECIMAL(18,2)) AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date
ORDER BY order_date"""
lab.sql(dashboard_sql, title="One row per day, for a dashboard")
expect(lab.query(dashboard_sql), [("2026-01-01", 5, 9, "1200.00", "133.33"), ("2026-01-02", 1, 1, "110.00", "110.00")],
       title="The daily rows and their averages")

<a id="unit-u.e44cdf968e204c85967b00d311cd5a54"></a>

**Күтілетін нәтиже**

Екі жол: 2026-01-01 күні 9-дың 5-еуі EAST тапсырысы, 1200.00, орташа 133.33; 2026-01-02 күні 1-дің 1-еуі EAST, 110.00, орташа 110.00. Жасыл белгі.

1200.00 / 9 — 133.333…; `CAST` көрсету үшін дөңгелектейді. Нақты мән қажет тұтынушы дөңгелектелген орташаны емес, қосындыларды алады.


<a id="unit-u.80bc6f474b8a483386eb0dc8a94b3965"></a>

<a id="3-reconcile-the-service-table-with-the-detail"></a>

## 3. Ұсыну кестесін егжей-тегжейлі деректермен салыстыру

Ұсыну кестесі қайта әрі тәуелсіз есептелген егжей-тегжейлі деректер сол сандарды бергенде дұрыс. Төмендегі сұрау келісімшартты `orders_fact_l2` ішінен есептейді (`detail`), ұсыну кестесін оқиды (`serving`), содан кейін ұсыну түйіршіктілігі — күн мен аймақ — бойынша біріктіреді. `FULL OUTER JOIN` бір жақта бар, екіншісінде жоқ жолды сақтайды, жетіспейтін жақта `NULL` болады.

Ұяшық салыстыруды үш рет орындайды: толық ұсыну кестесінде толық сыртқы біріктірумен; содан кейін сол күн жүктелмегендей 2 қаңтары жоқ ұсыну кестесінде екі рет — ішкі, толық сыртқы біріктірумен. Соңғы екеуінің қайсысы жетіспейтін күнді байқайтынын болжаңыз.


In [ ]:
reconcile_sql = """WITH detail AS (
    SELECT order_date, region, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
    FROM orders_fact_l2
    WHERE status <> 'CANCELLED'
    GROUP BY order_date, region
), serving AS (
    SELECT order_date, region, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2 {lost_day}
    GROUP BY order_date, region
)
SELECT CAST(COALESCE(d.order_date, s.order_date) AS STRING) AS order_date,
       COALESCE(d.region, s.region) AS region,
       d.order_count AS detail_orders, s.order_count AS serving_orders,
       d.gross_amount AS detail_amount, s.gross_amount AS serving_amount
FROM detail d
{join} serving s ON s.order_date = d.order_date AND s.region = d.region
ORDER BY order_date, region"""
complete = reconcile_sql.format(lost_day="", join="FULL OUTER JOIN")
lab.sql(complete, title="Detail and service table, full outer join")
expect(lab.query(complete), [("2026-01-01", "EAST", 5, 5, "720.00", "720.00"), ("2026-01-01", "WEST", 4, 4, "480.00", "480.00"),
                             ("2026-01-02", "EAST", 1, 1, "110.00", "110.00")], title="The service table matches the detail")
lost = "WHERE order_date < '2026-01-02'"
inner = reconcile_sql.format(lost_day=lost, join="INNER JOIN")
outer = reconcile_sql.format(lost_day=lost, join="FULL OUTER JOIN")
lab.sql(inner, title="A service table without January 2, inner join")
lab.sql(outer, title="A service table without January 2, full outer join")
expect([row for row in lab.query(inner) if row[2] != row[3] or row[4] != row[5]], [],
       title="The inner join finds no difference")
expect([row[:2] for row in lab.query(outer) if row[3] is None], [("2026-01-02", "EAST")],
       title="The full outer join shows the missing day")

<a id="unit-u.ff602d8664bd4d11842d3189a55667a8"></a>

**Күтілетін нәтиже**

Толық кестедегі толық сыртқы біріктіру: үш жол, әрқайсысында екі жақтағы тапсырыс пен сома бірдей, жасыл белгі. 2 қаңтарсыз ішкі біріктіру екеуі де тең екі жол көрсетеді, толық сыртқы біріктіру үшеуін, соңғысы егжей-тегжейлі деректерде 1 тапсырыс және 110.00, ұсыну кестесінде `NULL` бар 2026-01-02 EAST. Екі жасыл белгі: ішкі біріктіру айырма таппайды, толық сыртқы біріктіру жетіспейтін күнді көрсетеді.

Ішкі біріктіру тек екі жақта бар күндерді сақтады, сондықтан ұсыну кестесі жоғалтқан күн салыстыруды сәтсіз етудің орнына одан жоғалды. Себебін білмейінше `NULL` сақтаңыз: жоқ жол — нөл тапсырысы бар күн емес.


<a id="unit-u.cd7684126ce840eeb4bd7451cae61756"></a>

<a id="4-windows-and-ratios-on-the-real-days-and-regions"></a>

## 4. Нақты күндер мен аймақтардағы терезелер және қатынастар

Терезе функциясы байланысты жолдар бойынша есептеп, әр жолды сақтайды: `SUM(gross_amount) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` бірінші күннен ағымдағы күнге дейінгі сомаларды қосады, `LAG(gross_amount) OVER (ORDER BY order_date)` алдыңғы жол сомасын алады. CTE алдымен әр күнге бір жол жасайды, сондықтан терезелер тапсырыстар не аймақтармен емес, күндермен жұмыс істейді.

Сұрау 1 қаңтардың орташасын екі рет есептейді: екі аймақ орташасының орташасы және күннің қосындысын санына бөлу. Қайсысы күннің орташа тапсырыс сомасы екенін болжаңыз.


In [ ]:
window_sql = """WITH by_region AS (
    SELECT order_date, region, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2 GROUP BY order_date, region
), daily AS (
    SELECT order_date,
           SUM(order_count) AS order_count,
           SUM(gross_amount) AS gross_amount,
           CAST(AVG(gross_amount / order_count) AS DECIMAL(18,2)) AS average_of_region_averages
    FROM by_region GROUP BY order_date
)
SELECT CAST(order_date AS STRING) AS order_date, order_count, gross_amount,
       average_of_region_averages,
       CAST(gross_amount / order_count AS DECIMAL(18,2)) AS average_order_amount,
       SUM(gross_amount) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_amount,
       LAG(gross_amount) OVER (ORDER BY order_date) AS previous_day_amount
FROM daily
ORDER BY order_date"""
lab.sql(window_sql, title="Days, their averages, and two windows")
expect(lab.query(window_sql), [("2026-01-01", 9, "1200.00", "132.00", "133.33", "1200.00", None),
                               ("2026-01-02", 1, "110.00", "110.00", "110.00", "1310.00", "1200.00")],
       title="The windows keep one row per day; the average of averages is not the day's average")

<a id="unit-u.7f6689eca22c47d6b0f338304965594e"></a>

**Күтілетін нәтиже**

Екі жол және жасыл белгі. 2026-01-01: 9 тапсырыс, 1200.00, аймақ орташаларының орташасы 132.00, орташа тапсырыс сомасы 133.33, сол күнге дейін 1200.00, алдыңғы күн жоқ (`NULL`). 2026-01-02: 1 тапсырыс, 110.00, екі орташа да 110.00, сол күнге дейін 1310.00, алдыңғы күні 1200.00.

1 қаңтарда EAST-тің 5 тапсырысының орташасы 144.00, WEST-тің 4 тапсырысынікі 120.00. Жай орташа 132.00 төрт WEST тапсырысына бес EAST тапсырысындай салмақ береді; күн орташасы әр тапсырысты бір рет салмақтайды: 1200.00 / 9 = 133.33. Қосындылар мен сандарды сақтап, есеп түйіршіктілігінде бөліңіз. `LAG` алдыңғы жолды алды: мұнда барлық күн бар болғандықтан алдыңғы күн; бос күн болса, ескірек күн болар еді.


<a id="unit-u.11b985aad8334b298bff46d770d0a94c"></a>

<a id="5-from-order-amount-to-payment-metrics"></a>

## 5. Тапсырыс сомасынан төлем метрикаларына

Тапсырыс сомасы неге тапсырыс берілгенін айтады, не төленгенін емес. 9-жұмыс 5–7-бөлімдерінің мысалында төлемдер бар: сегіз тапсырысқа жеті төлем жеткізілімі, біреуі қайталау, біреуі есеп шегінен кейін және төлемнің бір бөлігін қайтару. Төмендегі метрикалар осы келісімшартқа сай; мұнда **тауарлардың жалпы құны (GMV)** төленген тапсырыстар құнын білдіреді.

| Метрика | Осы мысалдың келісімшарты |
| --- | --- |
| Төленген тауарлардың жалпы құны (GMV) | Asia/Shanghai бойынша `[January 1, January 4)` ішіндегі қайталанулары жойылған сәтті төлем сомаларының қосындысы; қайтарымдар бөлек |
| Төленген тапсырыстар / тұтынушылар | Сол төлем жиынындағы бөлек тапсырыс / тұтынушы ID-лері |
| Жасалған когорт конверсиясы | Сол терезеде жасалып, 4 қаңтарға дейін төленген тапсырыстарды терезеде жасалған барлық тапсырысқа бөлу |
| Қайта сатып алушы үлесі | Төлем терезесінде кемінде екі төленген тапсырысы бар тұтынушыларды төлем жасаған тұтынушыларға бөлу |
| Таза түсім | Төленген GMV минус сол терезеде алынған қайтарымдар; бухгалтерлік пайда емес |

Үлгіде әдейі **әр тапсырысқа бір сәтті төлем** бар. Бөліп төлеу жүйесіне төленген тапсырыстарды санамас бұрын тапсырыс деңгейінде басқа жинақтау қажет болар еді. Соңғы тапсырыс күйі төлем дәлелін алмастырмайды: ақшасы қайтарылған тапсырыс бастапқы төлемін бәрібір қосты.

Ұяшық қайтарым кестесін және әр төлем күніне бір жолы бар `paid_order_metrics_l2` жасап, терезе қорытындыларын егжей-тегжейлі деректермен салыстырады.


In [ ]:
case = case_fixture()
window_start, window_end, as_of = case["window_start"], case["window_end"], case["as_of"]
lab.execute("DROP VIEW IF EXISTS bi_paid_kpis_l2")
lab.execute("DROP TABLE IF EXISTS paid_order_metrics_l2")
lab.execute("DROP TABLE IF EXISTS dwd_refunds_l2")
lab.execute("""CREATE TABLE dwd_refunds_l2 (
    refund_id VARCHAR(32) NOT NULL, payment_id VARCHAR(32) NOT NULL,
    refund_amount DECIMAL(18,2) NOT NULL, refund_time DATETIME NOT NULL
) UNIQUE KEY(refund_id) DISTRIBUTED BY HASH(refund_id) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.insert("dwd_refunds_l2", ["refund_id","payment_id","refund_amount","refund_time"], case["refunds"])
lab.execute("""CREATE TABLE paid_order_metrics_l2 (
    payment_date DATE NOT NULL, paid_orders BIGINT NOT NULL,
    paid_gmv DECIMAL(18,2) NOT NULL, paid_customers BIGINT NOT NULL
) DUPLICATE KEY(payment_date) DISTRIBUTED BY HASH(payment_date) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.execute("""INSERT INTO paid_order_metrics_l2
SELECT DATE(p.payment_time), COUNT(DISTINCT p.order_id), SUM(p.amount), COUNT(DISTINCT o.customer_id)
FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
WHERE p.payment_time >= %s AND p.payment_time < %s
GROUP BY DATE(p.payment_time)""", (window_start, window_end))
expect(lab.query("SELECT CAST(payment_date AS STRING),paid_orders,paid_gmv,paid_customers FROM paid_order_metrics_l2 ORDER BY payment_date"), case["expected"]["payment_daily"],
       title="Three payment days, as the case expects")
# This is an independent detail calculation, not another read of the service result.
expect(lab.query("""SELECT COUNT(DISTINCT p.order_id), SUM(p.amount), COUNT(DISTINCT o.customer_id)
FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
WHERE p.payment_time >= %s AND p.payment_time < %s""", (window_start,window_end)), [(5,"660.00",3)],
       title="The window from the detail: 5 paid orders, 660.00, 3 customers")
lab.sql("SELECT * FROM paid_order_metrics_l2 ORDER BY payment_date", title="Paid orders per payment day");

<a id="unit-u.6b643d0992c345e498b5ce6b9c98bf3d"></a>

**Күтілетін нәтиже**

Екі жасыл белгі, содан кейін үш төлем күні: 2026-01-01 — 1 төленген тапсырыс, 100.00, 1 тұтынушы; 2026-01-02 — 2, 270.00, 2; 2026-01-03 — 2, 290.00, 1.

Терезеде 3 тұтынушыдан 5 төленген тапсырыс және 660.00 бар. PAY-B қайталауы бір рет саналады, себебі `dwd_payments_l2` әр төлем ID-іне бір жол сақтайды, 5 қаңтарда төленген PAY-E терезеден тыс. Күндік тұтынушы сандары 3 емес, 4 береді: 101-тұтынушы екі күнде төледі.


<a id="unit-u.ae4328b644394e0b9aa510afa8ecf6e0"></a>

<a id="6-reconcile-the-window-not-a-sum-of-daily-ratios"></a>

## 6. Күндік қатынастар қосындысын емес, терезені салыстыру

**Болжаңыз:** күндік бөлек сатып алушылар сандарын қоссақ төрт, бірақ терезеде тек үш бөлек сатып алушы бар. Бір тұтынушы екі күнде кездеседі. Сол сияқты 2 қаңтар төлемі 1 қаңтарда жасалған тапсырысқа тиесілі болуы мүмкін, сондықтан бір күннің төлем санын сол күннің жасалу санына бөлу когорт конверсиясы емес.

Егжей-тегжейлі деректерден бір жолдық тұтынушы келісімшартын жасаңыз. Ол алымдар мен бөлімдерді, айқын шекті сақтайды. 5 қаңтар төлемі тыс қалады; ішінара қайтарым төленген GMV-ді емес, таза түсімді азайтады. 11-жұмыстың қосымша бақылау тақтасы осы көріністі оқиды.


In [ ]:
# Timestamp strings come from the committed case fixture, not user-supplied SQL.
lab.execute(f"""CREATE VIEW bi_paid_kpis_l2 AS
WITH paid AS (
    SELECT p.order_id, o.customer_id, p.amount AS amount
    FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
    WHERE p.payment_time >= '{window_start}' AND p.payment_time < '{window_end}'
), buyer_orders AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS paid_orders FROM paid GROUP BY customer_id
), cohort AS (
    SELECT o.order_id, MAX(CASE WHEN p.payment_time < '{as_of}' THEN 1 ELSE 0 END) AS converted
    FROM dwd_orders_l2 o LEFT JOIN dwd_payments_l2 p ON o.order_id=p.order_id
    WHERE o.created_at >= '{window_start}' AND o.created_at < '{window_end}'
    GROUP BY o.order_id
), refunds AS (
    SELECT SUM(refund_amount) AS refunded FROM dwd_refunds_l2
    WHERE refund_time >= '{window_start}' AND refund_time < '{window_end}'
)
SELECT CAST('{as_of}' AS DATETIME) AS as_of,
       (SELECT SUM(amount) FROM paid) AS paid_gmv,
       (SELECT COUNT(DISTINCT order_id) FROM paid) AS paid_orders,
       (SELECT COUNT(*) FROM buyer_orders) AS paid_customers,
       (SELECT COUNT(*) FROM buyer_orders WHERE paid_orders >= 2) AS repeat_customers,
       (SELECT COUNT(*) FROM cohort) AS created_orders,
       (SELECT SUM(converted) FROM cohort) AS converted_orders,
       (SELECT SUM(converted)*1.0/NULLIF(COUNT(*),0) FROM cohort) AS cohort_conversion,
       (SELECT SUM(CASE WHEN paid_orders>=2 THEN 1 ELSE 0 END)*1.0/NULLIF(COUNT(*),0) FROM buyer_orders) AS repeat_buyer_rate,
       (SELECT refunded FROM refunds) AS refund_amount,
       (SELECT SUM(amount) FROM paid)-(SELECT refunded FROM refunds) AS net_collections
""")
expect(lab.query("SELECT paid_gmv,paid_orders,paid_customers,repeat_customers,created_orders,converted_orders,refund_amount,net_collections FROM bi_paid_kpis_l2"), [("660.00",5,3,2,8,5,"50.00","610.00")],
       title="The window's contract, from the detail")
expect(lab.query("SELECT SUM(paid_customers) FROM paid_order_metrics_l2"), [(4,)], title="The daily customer counts add up to 4, not to the window's 3")
lab.sql("SELECT * FROM bi_paid_kpis_l2", title="The payment contract for a dashboard");
cohort_members_sql = f"""SELECT o.order_id, o.status,
       MAX(CASE WHEN p.payment_time < '{as_of}' THEN 1 ELSE 0 END) AS converted
FROM dwd_orders_l2 o LEFT JOIN dwd_payments_l2 p ON o.order_id=p.order_id
WHERE o.created_at >= '{window_start}' AND o.created_at < '{window_end}'
GROUP BY o.order_id,o.status ORDER BY o.order_id"""
buyer_members_sql = f"""SELECT o.customer_id, COUNT(DISTINCT p.order_id) AS paid_orders
FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
WHERE p.payment_time >= '{window_start}' AND p.payment_time < '{window_end}'
GROUP BY o.customer_id ORDER BY o.customer_id"""
show_sql("Trace the ratios to their members", cohort_members_sql + ";\n" + buyer_members_sql)
cohort_members = lab.query(cohort_members_sql)
buyer_members = lab.query(buyer_members_sql)
show_records("Created cohort: all eight orders; converted=1 identifies the five paid before the cutoff",
             cohort_members, columns=["order_id","status","converted"], collapsed=True)
show_records("Paid buyers: all three customers; paid_orders>=2 identifies the two repeat buyers",
             buyer_members, columns=["customer_id","paid_orders"], collapsed=True)
expect((len(cohort_members), sum(row[2] for row in cohort_members),
        len(buyer_members), sum(row[1] >= 2 for row in buyer_members)), (8,5,3,2),
       title="The ratio members reconcile: 5 of 8 orders and 2 of 3 buyers")


<a id="unit-u.dbd7d123aff945b39210964361708c46"></a>

**Күтілетін нәтиже**

Үш жасыл белгі, содан кейін бір жол: 2026-01-04 00:00:00 бойынша 5 төленген тапсырыс пен 3 тұтынушыдан 660.00 төленген GMV, 2-еуі қайта сатып алушылар; терезеде 8 тапсырыс жасалған, 5-еуі шекке дейін төленген, `cohort_conversion` 0.62500; `repeat_buyer_rate` 0.66666 (3-тің 2-еуі); 50.00 қайтарылған; таза түсім 610.00.

Когорт терезеде жасалған 8 тапсырыстың бәрін, бас тартылған 2107-ні де санайды: ол жасалған, төленбеген, сондықтан конверсияланбаған. 1-бөлім тапсырыс метрикалары бас тартылғандарды тыс қалдырады; бұл келісімшарт қалдырмайтынын және себебін айтады. Екі қатынас та жолда алымы мен бөлімін сақтайды, оқырман тексере алады.
Екі мүшелер кестесін ашып, сегіз жасалған тапсырыс ID-ін, `converted=1` бар бесеуін, үш төлем жасаған тұтынушы ID-ін және кемінде екі бөлек төленген тапсырысы бар екеуін табыңыз. Бас тартылған 2107 конверсия бөлшегінің жасалған тапсырыстардан тұратын бөлімінде қалады; 5 қаңтардағы PAY-E шектен кейін және тапсырысын конверсияламайды. Бұл егжей-тегжейлі сұраулар жарияланған KPI көрінісін өзгертпей 8/5/3/2 сандарымен сәйкес келеді.


<a id="unit-u.3661e92bd5f6421b9d142931d8eabb85"></a>

<a id="7-exact-sets-and-approximate-sketches"></a>

## 7. Нақты жиындар және жуық ықшам көріністер

Нақты салыстыру HyperLogLog (HLL) бағасын үнсіз қабылдай алмайды. Bitmap агрегаттауы мұнда қолданылған үйлесімді теріс емес бүтін тұтынушы ID-лері үшін нақты. Бұл жасалған кірісте 100,000 жол, бірақ тек 20,000 бөлек ID бар; қайталанған ID-лер қосымша тұтынушыға айналмауы керек.

Нақты SQL жиынын, нақты Bitmap жиынын және HLL ықшам көрінісін **бірдей жиында** салыстырыңыз. Көрсетілген салыстырмалы қате — әмбебап қате шегі емес, бір бақылау. Үш тұтынушылық шағын бизнес мысалы бұл айырбасты түсіндіруге тым аз болар еді.


In [ ]:
distinct_counts = lab.query("""SELECT COUNT(DISTINCT customer_id),
    BITMAP_UNION_COUNT(TO_BITMAP(customer_id)), HLL_UNION_AGG(HLL_HASH(CAST(customer_id AS STRING)))
FROM (SELECT number % 20000 + 1 AS customer_id FROM numbers("number"="100000")) ids""")[0]
expect(distinct_counts[:2], (20000,20000), title="Two exact methods agree despite repeated IDs")
from dw_course.ui import show_records
show_records("Same-population distinct counts", [
    {"method":"COUNT(DISTINCT)", "count":distinct_counts[0], "exact":True},
    {"method":"Bitmap", "count":distinct_counts[1], "exact":True},
    {"method":"HLL", "count":distinct_counts[2], "exact":False,
     "observed_relative_error":abs(distinct_counts[2]-20000)/20000},
])

<a id="unit-u.017df6e866f440f596b67d4aea507c79"></a>

**Күтілетін нәтиже**

Жасыл белгі, екі нақты әдіс 20,000 мәнінде сәйкес келеді, содан кейін үш сан: `COUNT(DISTINCT)` және Bitmap 20,000 (нақты), HLL соған жақын 19,834, `observed_relative_error` 0.0083, 1 %-дан аз.

HLL әр ID орнына шағын ықшам көрініс сақтайды, сондықтан саны — бағалау; бір бақыланған қате басқа жиынға шек емес. Жуық сан жеткілікті және тұтынушылар оның жуық екенін білетін жерде қолданыңыз.


<a id="unit-u.79443f91485b42c881128e6eb973a64d"></a>

<a id="8-measure-a-result-preserving-optimization"></a>

## 8. Нәтижені сақтайтын оңтайландыруды өлшеу

32 күнге 100,000 егжей-тегжейлі тапсырыс жасап, содан кейін 32 жолдық күндік ұсыну кестесін қайта құрыңыз. Екі сұрау **әр күнді, санды және соманы бірдей** қайтаруы керек. Тек содан кейін нақты Query Profile-дарын салыстырыңыз.

Көмекші өлшенген Profile ID-лері мен сканерлеу есептегіштерін көрсетеді. Ол сондай-ақ бір хостта, параллельдігі бір, бір қыздыру орындалуынан кейін, SQL нәтиже кэштері өшірулі әр сұраудың 20 кідіріс үлгісін алады. Өзгерген сеанс айнымалыларының бәрін қалпына келтіреді. Сақтау кэштері тазаланбайды. Эмпирикалық 95-процентиль (P95) — көрсетілім: бұл бір түйінді сынақ бақылау тақтасының қызмет деңгейі келісімін (SLA), көп пайдаланушы мінез-құлқын не өндірістік жылдамдауды дәлелдемейді.


In [ ]:
lab.execute("DROP TABLE IF EXISTS perf_orders_detail_l2")
lab.execute("DROP TABLE IF EXISTS perf_daily_orders_l2")
lab.execute("""CREATE TABLE perf_orders_detail_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date,order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.execute("""INSERT INTO perf_orders_detail_l2
SELECT DATE_ADD(CAST('2026-01-01' AS DATE), CAST(number%32 AS INT)), number, number%100+1
FROM numbers("number"="100000")""")
lab.execute("""CREATE TABLE perf_daily_orders_l2 (
    order_date DATE NOT NULL, order_count BIGINT SUM NOT NULL,
    amount DECIMAL(18,2) SUM NOT NULL
) AGGREGATE KEY(order_date) DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
# Rebuild, rather than appending the same additive batch twice.
lab.execute("INSERT INTO perf_daily_orders_l2 SELECT order_date,COUNT(*),SUM(amount) FROM perf_orders_detail_l2 GROUP BY order_date")
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM perf_orders_detail_l2"), [(100000,"5050000.00")])
expect(lab.query("SELECT SUM(order_count),SUM(amount) FROM perf_daily_orders_l2"), [(100000,"5050000.00")],
       title="The 32-row table holds the same 100,000 orders and 5,050,000.00")
profile_evidence = compare_query_profiles(lab, [
    ("Detail aggregation", "SELECT order_date,COUNT(*) AS orders,SUM(amount) AS amount FROM perf_orders_detail_l2 GROUP BY order_date ORDER BY order_date"),
    ("Daily service aggregation", "SELECT order_date,SUM(order_count) AS orders,SUM(amount) AS amount FROM perf_daily_orders_l2 GROUP BY order_date ORDER BY order_date"),
]);

<a id="unit-u.aea6b35b6bc24f15b164fc363b56b2e2"></a>

**Күтілетін нәтиже**

Жасыл белгі, содан кейін екі сұраудың бірдей 32 жол қайтаратынына жасыл белгі, екі толық Query Profile (жиналған күйде) және сканерлеу дәлелі: егжей-тегжейлі агрегаттау 100,000 жол, ұсыну агрегаттауы 32 оқиды. Содан кейін кідіріс үлгілері: әр сұрау бір-бірден 20 рет, әрқайсысының эмпирикалық P95 мәні бірнеше миллисекунд; ұсыну кестесі әдетте жылдамырақ, бірақ осы шағын кестеде айырма аз, орындаудан орындауға өзгеруі мүмкін.

Алдымен нәтижелер теңдігі: басқа жауап беретін жылдамырақ сұрау оңтайландыру емес. Profile есептегіштері әр орындалудың не оқығанын көрсетеді; уақыттар өндірістік бақылау тақтасына емес, осы компьютер мен осы сәтке тиесілі.


<a id="unit-u.d77b6579b0ff4f03958870534027c4db"></a>

<a id="9-practice-orders-per-customer-and-day"></a>

## 9. Жаттығу: тұтынушы мен күн бойынша тапсырыстар

Бизнес әр тұтынушы мен күннің тапсырыстары мен сомасын сұрайды. `daily_order_metrics_l2` жауап бере алмайды: жолдарында тұтынушы жоқ, еш сұрау оны қайтара алмайды. Сол келісімшартпен `orders_fact_l2` ішінен тұтынушы-күн түйіршіктілігін есептеп, жолдарын көрсетіңіз және егжей-тегжейлі деректердің 10 тапсырысы мен 1310.00 сомасын беретінін тексеріңіз. `daily_order_metrics_l2` өзгермесін: 11-жұмыс оны оқиды.


In [ ]:
# Your code here

<a id="unit-u.1eba5ed30b9f409190dc65247ff94927"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
customer_day_sql = """SELECT order_date, customer_id, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_fact_l2
WHERE status <> 'CANCELLED'
GROUP BY order_date, customer_id"""
lab.sql(customer_day_sql + " ORDER BY order_date, customer_id", title="Orders per customer and day")
expect(lab.query(f"SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM ({customer_day_sql}) AS customer_days"),
       [(10, 10, "1310.00")], title="Ten customer-days, 10 orders, 1310.00")
expect(lab.query("SELECT SUM(order_count), SUM(gross_amount) FROM daily_order_metrics_l2"), [(10, "1310.00")],
       title="The service table is unchanged")

<a id="unit-u.192bb8847f314db3a7d3c484d7e5c383"></a>

**Күтілетін нәтиже**

Он жол, әр тұтынушы мен күнге біреуден: 2026-01-01 күні 1 және 3–10 тұтынушылар, 2026-01-02 күні 11-тұтынушы, әрқайсысында 50.00-ден (9-тұтынушы) 300.00-ге дейінгі (7-тұтынушы) бір тапсырыс. Екі жасыл белгі: он тұтынушы-күн, 10 тапсырыс, 1310.00 және ұсыну кестесінде әлі 10 тапсырыс, 1310.00.

Бұл деректерде әр тұтынушының бір тапсырысы бар, сондықтан тұтынушы-күн жолдары тапсырыстармен бірдей санды. Ұсыну кестесі сақтамайтын түйіршіктілік егжей-тегжейлі деректерден алынады; оның орнына әр тұтынушыға күннің жалпы сомасын беру күнді әр тұтынушыға бір рет санар еді.


<a id="unit-u.337e2e106b26429eb15c85ebbf13e87f"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- `daily_order_metrics_l2` келісімшартын: жиын, түйіршіктілік, көрсеткіштер, уақыт және қатынас, неге қосындылар мен сандар сақтайтынын айта аласыз;
- ұсыну кестесі толық сыртқы біріктіруде егжей-тегжейлі деректерге сәйкес, ішкі біріктіру неге жоғалған күнді байқамағанын айта аласыз;
- аймақ орташаларының орташасы 132.00 неге 1 қаңтардың орташа тапсырыс сомасы 133.33 емес екенін түсіндіре аласыз;
- мысалдың төленген GMV 660.00 мәнін тапсырыс сомасынан ажыратып, `bi_paid_kpis_l2` әр қатынасы не санайтынын айта аласыз;
- тең нәтижелер, Profile есептегіштері және уақыттар ұсыну кестесі туралы әрқайсысы не көрсететінін айта аласыз.


<a id="unit-u.ccde06de15ae4e128d2eb3116b02e13d"></a>

<a id="next"></a>

## Келесі қадам

[10-тест: метрикаларды өңдеу және ұсыну](quiz10_metric_processing.ipynb) файлын ашыңыз, содан кейін [11-модуль: BI және AI тұтынушыларына арналған интерфейстер мен шекаралар](../module11-bi-and-ai/course.md) бөліміне өтіңіз.
